In [1]:
import os
import time
import pandas as pd
from dotenv import load_dotenv
import openai
import anthropic
from google import genai
from google.genai import types
import mlflow

load_dotenv()

if "GOOGLE_API_KEY" in os.environ:
    os.environ.pop("GOOGLE_API_KEY")

openai_client = openai.OpenAI(api_key=os.getenv("OPENAI_API_KEY"))
anthropic_client = anthropic.Anthropic(api_key=os.getenv("ANTHROPIC_API_KEY"))
gemini_client = genai.Client(api_key=os.getenv("GEMINI_API_KEY"))

local_path = "../data/FinancialPhraseBank-v1.0/Sentences_AllAgree.txt"  # FIX: relative path so this runs on any machine

print("Loading Financial PhraseBank dataset...")
samples = []
with open(local_path, encoding="latin-1") as f:
    for line in f.readlines()[:100]:
        line = line.strip()
        if "@" in line:
            sentence, label = line.rsplit("@", 1)
            samples.append({"sentence": sentence.strip(), "label": label.strip().lower()})

print(f"Loaded {len(samples)} samples")

SYSTEM_PROMPT = (
    "You are a financial sentiment classifier. "
    "Classify the sentiment as EXACTLY one of: negative, neutral, positive. "
    "Reply with ONE word only."
)

MAX_RETRIES = 3  # retry transient API errors (rate limits, 5xx, timeouts) before giving up on a sample

def classify(client_type, text):
    if client_type == "openai":
        last_exc = None
        for attempt in range(MAX_RETRIES):
            start = time.time()
            try:
                r = openai_client.chat.completions.create(
                    model="gpt-4o",  # FIX
                    messages=[
                        {"role": "system", "content": SYSTEM_PROMPT},
                        {"role": "user", "content": text}
                    ],
                    max_tokens=5,
                    temperature=0.0  # FIX
                )
                return r.choices[0].message.content.strip().lower(), time.time() - start
            except Exception as e:
                last_exc = e
                if attempt < MAX_RETRIES - 1:
                    wait = (attempt + 1) * 3
                    print(f"OpenAI error, retrying in {wait}s (attempt {attempt + 1}/{MAX_RETRIES}): {e}")
                    time.sleep(wait)
        print(f"OpenAI error after retries: {last_exc}")
        return "error", time.time() - start

    elif client_type == "anthropic":
        last_exc = None
        for attempt in range(MAX_RETRIES):
            start = time.time()
            try:
                r = anthropic_client.messages.create(
                    model="claude-haiku-4-5-20251001",
                    max_tokens=5,
                    temperature=0.0,  # FIX: match GPT-4o/Gemini's greedy decoding for a fair comparison
                    system=SYSTEM_PROMPT,
                    messages=[{"role": "user", "content": text}]
                )
                return r.content[0].text.strip().lower(), time.time() - start
            except Exception as e:
                last_exc = e
                if attempt < MAX_RETRIES - 1:
                    wait = (attempt + 1) * 3
                    print(f"Anthropic error, retrying in {wait}s (attempt {attempt + 1}/{MAX_RETRIES}): {e}")
                    time.sleep(wait)
        print(f"Anthropic error after retries: {last_exc}")
        return "error", time.time() - start

    elif client_type == "gemini":
        for attempt in range(4):
            start = time.time()  # FIX: reset per attempt so retry backoff sleep isn't counted as latency
            try:
                r = gemini_client.models.generate_content(
                    model="gemini-2.5-flash",
                    contents=f"{SYSTEM_PROMPT}\n\nText: {text}",
                    config=types.GenerateContentConfig(
                        max_output_tokens=5,
                        temperature=0.0,
                        thinking_config=types.ThinkingConfig(thinking_budget=0)
                    )
                )
                return r.text.strip().lower() if r.text else "unknown", time.time() - start
            except Exception as e:
                if "503" in str(e) and attempt < 3:
                    time.sleep((attempt + 1) * 3)
                    continue
                print(f"Gemini error: {e}")
                return "error", time.time() - start

# Fixed (non-timestamped) path so an interrupted run can resume instead of
# losing already-paid-for API calls.
os.makedirs("../logs", exist_ok=True)
csv_path = "../logs/financial_baseline.csv"

if os.path.exists(csv_path):
    try:
        loaded = pd.read_csv(csv_path)
        if loaded.empty or "openai_correct" not in loaded.columns:
            raise ValueError("checkpoint has an unexpected/empty schema")
        results = loaded.to_dict("records")
        print(f"Resuming: {len(results)}/{len(samples)} samples already done")
    except Exception as e:
        print(f"Ignoring unusable checkpoint at {csv_path} ({e}) — starting fresh")
        results = []
else:
    results = []

mlflow.set_experiment("financial_sentiment_baseline")

with mlflow.start_run(run_name="sentiment_baseline_v2_fixed"):
    mlflow.log_param("sample_count", 100)
    mlflow.log_param("openai_model", "gpt-4o")
    mlflow.log_param("anthropic_model", "claude-haiku-4-5-20251001")
    mlflow.log_param("gemini_model", "gemini-2.5-flash")

    print("Evaluating...")
    for i, sample in enumerate(samples):
        if i < len(results):
            continue  # already done in a previous (interrupted) run

        text = sample["sentence"]
        true_label = sample["label"]

        oai_pred, oai_lat = classify("openai", text)
        ant_pred, ant_lat = classify("anthropic", text)
        gem_pred, gem_lat = classify("gemini", text)

        results.append({
            "text":              text[:80],
            "true_label":        true_label,
            "openai_pred":       oai_pred,
            "anthropic_pred":    ant_pred,
            "gemini_pred":       gem_pred,
            "openai_correct":    int(oai_pred == true_label),
            "anthropic_correct": int(ant_pred == true_label),
            "gemini_correct":    int(gem_pred == true_label),
            "openai_latency_s":  round(oai_lat, 3),
            "anthropic_latency_s": round(ant_lat, 3),
            "gemini_latency_s":  round(gem_lat, 3),
        })

        if (i + 1) % 20 == 0:
            pd.DataFrame(results).to_csv(csv_path, index=False)  # checkpoint

        if i % 10 == 0:
            print(f"Progress: {i}/100")

    # Metrics — derived from `results`, so they're correct whether this run
    # processed everything fresh or resumed a partial checkpoint.
    df = pd.DataFrame(results)
    oai_acc = df["openai_correct"].mean()
    ant_acc = df["anthropic_correct"].mean()
    gem_acc = df["gemini_correct"].mean()

    mlflow.log_metric("openai_accuracy",        oai_acc)
    mlflow.log_metric("anthropic_accuracy",     ant_acc)
    mlflow.log_metric("gemini_accuracy",        gem_acc)
    mlflow.log_metric("openai_avg_latency_s",   df["openai_latency_s"].mean())
    mlflow.log_metric("anthropic_avg_latency_s",df["anthropic_latency_s"].mean())
    mlflow.log_metric("gemini_avg_latency_s",   df["gemini_latency_s"].mean())

    print(f"\n{'='*55}")
    print(f"{'Metric':<22} {'GPT-4o':>8} {'Haiku':>10} {'Gemini':>10}")
    print(f"{'='*55}")
    print(f"{'Avg Latency (s)':<22} {df['openai_latency_s'].mean():>8.3f} {df['anthropic_latency_s'].mean():>10.3f} {df['gemini_latency_s'].mean():>10.3f}")
    print(f"{'Accuracy':<22} {oai_acc:>8.2%} {ant_acc:>10.2%} {gem_acc:>10.2%}")
    print(f"{'='*55}")

pd.DataFrame(results).to_csv(csv_path, index=False)  # final save
print(f"Saved to {csv_path}")


Loading Financial PhraseBank dataset...
Loaded 100 samples
Evaluating...
Progress: 0/100
Progress: 10/100
Progress: 20/100
Progress: 30/100
Progress: 40/100
Progress: 50/100
Progress: 60/100
Progress: 70/100
Progress: 80/100
Progress: 90/100

Metric                   GPT-4o      Haiku     Gemini
Avg Latency (s)           0.735      0.863      0.554
Accuracy                 99.00%     93.00%     99.00%
Saved to ../logs/financial_baseline_20260526_134703.csv
